In [2]:
import pandas as pd
import joblib
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline


# -----------------------------
# Step 1: Load Train Dataset
# -----------------------------
file_path = r"train.csv"
df = pd.read_csv(file_path)

# -----------------------------
# Step 2: Preprocessor Setup
# -----------------------------
# Define Column Groups
text_cols = ["title", "description", "requirements", "company_profile", "benefits"]
cat_cols  = ["location", "department", "employment_type", "required_experience", "required_education", "industry", "function"]
num_cols  = ["telecommuting", "has_company_logo", "has_questions"]

# Clean Text & Categorical Missing Values
for col in text_cols:
    df[col] = df[col].replace({None: ""}).fillna("").astype(str)

for col in cat_cols:
    df[col] = df[col].replace({None: "Unspecified"}).fillna("Unspecified").astype(str)

for col in num_cols:
    df[col] = df[col].fillna(0).astype(int)

# Target and Feature Matrices
X_train = df[text_cols + cat_cols + num_cols]
y_train = df["fraudulent"]

print(f"Training Records: {df.shape[0]}")
print(f"Dataset Columns : {df.shape[1]}")

preprocessor = ColumnTransformer(
    transformers=[
        ("title",    TfidfVectorizer(max_features=500, stop_words="english"), "title"),
        ("desc",     TfidfVectorizer(max_features=1500, stop_words="english"), "description"),
        ("req",      TfidfVectorizer(max_features=1000, stop_words="english"), "requirements"),
        ("profile",  TfidfVectorizer(max_features=500, stop_words="english"), "company_profile"),
        ("benefits", TfidfVectorizer(max_features=500, stop_words="english"), "benefits"),
        ("cat",      OneHotEncoder(handle_unknown="ignore", min_frequency=5), cat_cols),
        ("num",      "passthrough", num_cols),
    ],
    remainder="drop"
)

# -----------------------------
# Step 3: Model Pipeline
# -----------------------------
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("resampler", SMOTE(random_state=42)),
    ("classifier",
     SVC(kernel="linear", probability=True, random_state=42))
])

# -----------------------------
# Step 4: Train the Model
# -----------------------------

print("Original class distribution:", Counter(y_train))

# Train model
pipeline.fit(X_train, y_train)

## Show transformed feature matrix size
X_processed = pipeline.named_steps["preprocessor"].transform(X_train)
print("Feature Matrix Shape:", X_processed.shape)

#save model
joblib.dump(pipeline, "decision_tree_model.pkl")

joblib.dump(
    {
        "model": pipeline,
        "text_cols": text_cols,
        "cat_cols": cat_cols,
        "num_cols": num_cols
    },
    "support_vector_machine_model.pkl"
)

print("Training completed!")

print(
    "Model saved as support_vector_machine_model.pkl"
)

Training Records: 14304
Dataset Columns : 18
Original class distribution: Counter({0: 13619, 1: 685})


C:\Users\jpalaniappan\AppData\Local\anaconda3\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Feature Matrix Shape: (14304, 4777)
Training completed!
Model saved as support_vector_machine_model.pkl
